# 02 — Q1: Phương pháp sơ chế và chất lượng

**Người phụ trách:** Ngọc Hoàng · **Tiêu chí:** D3, D4, D5 · Nguồn: `docs/questions.md`

**Câu hỏi:** Trong cùng một châu lục, các lô sơ chế khô (*Natural / Dry*) có `total_9` cao hơn các lô sơ chế ướt (*Washed / Wet*) không, chênh lệch bao nhiêu điểm (CI 95%), và chênh lệch thô trên toàn mẫu có đổi chiều sau khi kiểm soát châu lục không?

- **Motivation:** nông hộ và hợp tác xã cân nhắc chuyển quy trình sơ chế (sơ chế khô tốn ít nước hơn) cần biết việc chuyển đổi có làm giảm điểm chất lượng không.
- **Phạm vi:** các lô có đủ châu lục và phương pháp sơ chế; chỉ giữ 2 phương pháp trên; chỉ giữ châu lục có ≥ 20 lô mỗi nhóm.
- **Bằng chứng:** D3 `total_9` (có biện luận) · D4 bảng chênh lệch thô vs chênh lệch trong từng châu lục · D5 bootstrap CI 95% + Cohen's d.

> Notebook này chỉ gồm phần **Q1 preprocessing** (luồng lọc dữ liệu → `data/processed/q1_clean.csv`).

<div style="background-color:#283E75;color:#ffffff;padding:14px 24px;border-radius:8px;margin-top:24px">

<span style="font-size:17px;font-weight:700;letter-spacing:3px;color:#E3EBFA">PHẦN 0</span>

## <span style="color:#ffffff">Môi trường code &amp; thư viện</span>

<span style="font-size:15px;color:#DCE6F7">Kiểm tra kernel và phiên bản thư viện khớp với <b>environment.yml</b></span>

</div>

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display

NOTEBOOK_CWD = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_CWD.parent if NOTEBOOK_CWD.name == "notebooks" else NOTEBOOK_CWD
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import PROCESSED, SCORE_COLS
from src.cleaning import load_raw, convert_types, flag_placeholders
from src.features import add_total_score
from src.labels import normalize_labels

pd.set_option("display.max_columns", None)

%matplotlib inline

print("python    :", sys.executable)
print("numpy     :", np.__version__)
print("pandas    :", pd.__version__)
print("matplotlib:", matplotlib.__version__)

python    : /home/hoang/miniconda3/envs/coffee/bin/python
numpy     : 1.26.0
pandas    : 2.1.1
matplotlib: 3.8.0


<span style="color:#5B6B8C;font-size:14px">→ Mong đợi: đường dẫn kết thúc bằng <code>.../envs/coffee/bin/python</code></span>

In [2]:
# TEST
def major_minor(version):
    """'2.1.1' -> '2.1'"""
    return ".".join(version.split(".")[:2])

assert major_minor(np.__version__) == "1.26", f"numpy {np.__version__} khác environment.yml"
assert major_minor(pd.__version__) == "2.1", f"pandas {pd.__version__} khác environment.yml"
assert major_minor(matplotlib.__version__) == "3.8", f"matplotlib {matplotlib.__version__} khác environment.yml"
print("Môi trường OK")

Môi trường OK


<div style="background-color:#283E75;color:#ffffff;padding:14px 24px;border-radius:8px;margin-top:24px">

<span style="font-size:17px;font-weight:700;letter-spacing:3px;color:#E3EBFA">PHẦN 1</span>

## <span style="color:#ffffff">Q1 preprocessing</span>

<span style="font-size:15px;color:#DCE6F7">Lọc mẫu phân tích cho Q1 và ghi lại số lô sau từng bước</span>

</div>

### <span style="color:#283E75">▍Luồng tiền xử lý</span>

1. Đọc dữ liệu → `convert_types` → `normalize_labels` → `add_total_score`.
2. Chỉ giữ loài Arabica.
3. Bỏ bản ghi giữ chỗ `flag_placeholders(...)["zero_all_scores"]` (`REC_ID = 1310`).
4. Bỏ lô thiếu `Continent.of.Origin` hoặc `Processing.Method`.
5. In `value_counts()` của `Processing.Method` để lấy đúng chuỗi nhãn sơ chế ướt (*Washed / Wet*) và khô (*Natural / Dry*) sau chuẩn hoá.
6. Chỉ giữ 2 phương pháp sơ chế trên.
7. Chỉ giữ châu lục có ≥ 20 lô ở **mỗi** phương pháp (crosstab Châu lục × Phương pháp trước / sau khi lọc).
8. Giữ cả cặp `REC_ID` 721 / 742 và lô `REC_ID = 1306` (theo `docs/questions.md`).

In [3]:
CONTINENT_COL = "Continent.of.Origin"
METHOD_COL = "Processing.Method"
MIN_PER_GROUP = 20        # ngưỡng ≥ 20 lô mỗi nhóm (docs/questions.md)

row_counts = {}           # Bước -> số lô còn lại sau bước đó

### <span style="color:#283E75">▍Bước 1 · Đọc dữ liệu, ép kiểu, chuẩn hoá nhãn, tính <code>total_9</code></span>

In [4]:
df_raw = load_raw()
df = add_total_score(normalize_labels(convert_types(df_raw)))

row_counts["1. Dữ liệu gốc (load → convert → normalize → total_9)"] = len(df)
print(f"Số lô: {len(df):,} · số cột: {df.shape[1]}")

Số lô: 1,339 · số cột: 23


### <span style="color:#283E75">▍Bước 2 · Chỉ giữ Arabica</span>

<span style="color:#5B6B8C;font-size:14px"><code>normalize_labels</code> đưa nhãn về chữ thường (<code>casefold</code>) → in nhãn thực tế trước khi lọc.</span>

In [5]:
print(df["Species"].value_counts(dropna=False))

Species
arabica    1311
robusta      28
Name: count, dtype: Int64


In [6]:
ARABICA = "arabica"       # đúng chuỗi in ra ở ô trên (sau casefold)
assert ARABICA in set(df["Species"].dropna()), f"Không tìm thấy nhãn {ARABICA!r} trong Species"

df_q1 = df[df["Species"].eq(ARABICA)].copy()
row_counts["2. Chỉ giữ Arabica"] = len(df_q1)
print(f"Còn {len(df_q1):,} lô (bỏ {len(df) - len(df_q1):,})")

Còn 1,311 lô (bỏ 28)


### <span style="color:#283E75">▍Bước 3 · Bỏ bản ghi giữ chỗ (<code>REC_ID = 1310</code>)</span>

In [7]:
placeholder = flag_placeholders(df_q1)["zero_all_scores"]
print("REC_ID bị đánh dấu zero_all_scores:", df_q1.loc[placeholder, "REC_ID"].tolist())

before = len(df_q1)
df_q1 = df_q1[~placeholder].copy()
row_counts["3. Bỏ bản ghi giữ chỗ (zero_all_scores)"] = len(df_q1)
print(f"Còn {len(df_q1):,} lô (bỏ {before - len(df_q1):,})")

REC_ID bị đánh dấu zero_all_scores: [1310]
Còn 1,310 lô (bỏ 1)


### <span style="color:#283E75">▍Bước 4 · Bỏ lô thiếu châu lục hoặc phương pháp sơ chế</span>

In [8]:
print("Số lô thiếu theo cột:")
print(df_q1[[CONTINENT_COL, METHOD_COL]].isna().sum())

before = len(df_q1)
df_q1 = df_q1.dropna(subset=[CONTINENT_COL, METHOD_COL]).copy()
row_counts["4. Bỏ lô thiếu châu lục / phương pháp sơ chế"] = len(df_q1)
print(f"\nCòn {len(df_q1):,} lô (bỏ {before - len(df_q1):,})")

Số lô thiếu theo cột:
Continent.of.Origin      1
Processing.Method      151
dtype: int64

Còn 1,159 lô (bỏ 151)


### <span style="color:#283E75">▍Bước 5 · Xác định đúng chuỗi nhãn phương pháp sơ chế</span>

In [9]:
print(df_q1[METHOD_COL].value_counts(dropna=False))

Processing.Method
washed / wet                 812
natural / dry                251
semi-washed / semi-pulped     56
other                         26
pulped natural / honey        14
Name: count, dtype: Int64


In [10]:
# Đúng chuỗi in ra ở ô trên (sau normalize_labels), không đoán.
WET = "washed / wet"      # sơ chế ướt (Washed / Wet)
DRY = "natural / dry"     # sơ chế khô (Natural / Dry)
Q1_METHODS = [WET, DRY]

observed_methods = set(df_q1[METHOD_COL].dropna())
missing_labels = [m for m in Q1_METHODS if m not in observed_methods]
assert not missing_labels, f"Không tìm thấy nhãn {missing_labels} trong {sorted(observed_methods)}"
print("Nhãn dùng cho Q1:", Q1_METHODS)

row_counts["5. Xác định nhãn sơ chế (không lọc)"] = len(df_q1)

Nhãn dùng cho Q1: ['washed / wet', 'natural / dry']


### <span style="color:#283E75">▍Bước 6 · Chỉ giữ 2 phương pháp sơ chế</span>

In [11]:
before = len(df_q1)
df_q1 = df_q1[df_q1[METHOD_COL].isin(Q1_METHODS)].copy()
row_counts["6. Chỉ giữ washed / wet và natural / dry"] = len(df_q1)
print(f"Còn {len(df_q1):,} lô (bỏ {before - len(df_q1):,})")

Còn 1,063 lô (bỏ 96)


### <span style="color:#283E75">▍Bước 7 · Chỉ giữ châu lục có ≥ 20 lô ở mỗi phương pháp</span>

<span style="color:#5B6B8C;font-size:14px"><b>Trước khi lọc</b> · cột <code>min_per_group</code> = số lô của nhóm ít hơn trong châu lục.</span>

In [12]:
ct_before = pd.crosstab(df_q1[CONTINENT_COL], df_q1[METHOD_COL]).reindex(columns=Q1_METHODS, fill_value=0)
ct_before["min_per_group"] = ct_before[Q1_METHODS].min(axis=1)
ct_before["keep"] = ct_before["min_per_group"].ge(MIN_PER_GROUP)
display(ct_before)

Processing.Method,washed / wet,natural / dry,min_per_group,keep
Continent.of.Origin,,,,
africa,100,27,27,True
asia,86,27,27,True
north america,490,89,89,True
oceania,1,0,0,False
south america,135,108,108,True


In [13]:
kept_continents = ct_before.index[ct_before["keep"]].tolist()
dropped_continents = ct_before.index[~ct_before["keep"]].tolist()
print("Giữ :", kept_continents)
print("Loại:", dropped_continents)

before = len(df_q1)
df_q1 = df_q1[df_q1[CONTINENT_COL].isin(kept_continents)].copy()
row_counts["7. Chỉ giữ châu lục có ≥ 20 lô mỗi phương pháp"] = len(df_q1)
print(f"Còn {len(df_q1):,} lô (bỏ {before - len(df_q1):,})")

Giữ : ['africa', 'asia', 'north america', 'south america']
Loại: ['oceania']
Còn 1,062 lô (bỏ 1)


<span style="color:#5B6B8C;font-size:14px"><b>Sau khi lọc</b></span>

In [14]:
ct_after = pd.crosstab(df_q1[CONTINENT_COL], df_q1[METHOD_COL]).reindex(columns=Q1_METHODS, fill_value=0)
display(ct_after)

Processing.Method,washed / wet,natural / dry
Continent.of.Origin,,
africa,100,27
asia,86,27
north america,490,89
south america,135,108


### <span style="color:#283E75">▍Bước 8 · Giữ cặp <code>REC_ID</code> 721 / 742 và lô 1306</span>

<span style="color:#5B6B8C;font-size:14px">Không lọc gì ở bước này; chỉ xác nhận các bản ghi vẫn còn trong mẫu.</span>

In [15]:
KEEP_IDS = [721, 742, 1306]
display(df_q1.loc[df_q1["REC_ID"].isin(KEEP_IDS),
                  ["REC_ID", CONTINENT_COL, "Country.of.Origin", METHOD_COL, "Clean.Cup", "total_9"]])

row_counts["8. Giữ REC_ID 721 / 742 và 1306 (không lọc)"] = len(df_q1)

,REC_ID,Continent.of.Origin,Country.of.Origin,Processing.Method,Clean.Cup,total_9
721,721,asia,taiwan,washed / wet,10.0,75.00
742,742,asia,taiwan,washed / wet,10.0,75.00
1306,1306,north america,mexico,washed / wet,0.0,61.58


### <span style="color:#283E75">▍Tóm tắt số lô sau từng bước</span>

In [16]:
summary = pd.DataFrame(list(row_counts.items()), columns=["Bước", "Số lô"])
display(summary)

,Bước,Số lô
0,1. Dữ liệu gốc (load → convert → normalize → t...,1339
1,2. Chỉ giữ Arabica,1311
2,3. Bỏ bản ghi giữ chỗ (zero_all_scores),1310
3,4. Bỏ lô thiếu châu lục / phương pháp sơ chế,1159
4,5. Xác định nhãn sơ chế (không lọc),1159
5,6. Chỉ giữ washed / wet và natural / dry,1063
6,7. Chỉ giữ châu lục có ≥ 20 lô mỗi phương pháp,1062
7,8. Giữ REC_ID 721 / 742 và 1306 (không lọc),1062


In [17]:
final_crosstab = pd.crosstab(
    df_q1[CONTINENT_COL], df_q1[METHOD_COL], margins=True, margins_name="Total",
).reindex(columns=Q1_METHODS + ["Total"])
display(final_crosstab)

Processing.Method,washed / wet,natural / dry,Total
Continent.of.Origin,,,
africa,100,27,127
asia,86,27,113
north america,490,89,579
south america,135,108,243
Total,811,251,1062


In [18]:
# TEST
methods_left = set(df_q1[METHOD_COL].unique())
assert methods_left == set(Q1_METHODS), f"Phải còn đúng 2 phương pháp {Q1_METHODS}, đang có {sorted(methods_left)}"

ct_check = pd.crosstab(df_q1[CONTINENT_COL], df_q1[METHOD_COL]).reindex(columns=Q1_METHODS, fill_value=0)
too_small = ct_check[ct_check.min(axis=1).lt(MIN_PER_GROUP)]
assert too_small.empty, f"Có châu lục < {MIN_PER_GROUP} lô ở một phương pháp:\n{too_small}"

assert not df_q1["REC_ID"].eq(1310).any(), "REC_ID 1310 (bản ghi giữ chỗ) vẫn còn trong mẫu"
assert df_q1["Species"].eq(ARABICA).all(), f"Còn loài khác Arabica: {df_q1['Species'].unique().tolist()}"

missing_ids = sorted(set(KEEP_IDS) - set(df_q1["REC_ID"]))
assert not missing_ids, f"Thiếu REC_ID cần giữ: {missing_ids}"

assert df_q1["total_9"].notna().all(), f"total_9 có {df_q1['total_9'].isna().sum()} giá trị NaN"

assert len(df_q1) == summary["Số lô"].iloc[-1], (
    f"Số dòng cuối {len(df_q1)} khác dòng cuối bảng tóm tắt {summary['Số lô'].iloc[-1]}"
)
print("Q1 preprocessing checks passed")

Q1 preprocessing checks passed


### <span style="color:#283E75">▍D4 · So sánh thô và so sánh có kiểm soát châu lục</span>

<span style="color:#5B6B8C;font-size:14px">Mọi chênh lệch tính theo thứ tự <b>natural − washed</b>, đơn vị điểm <code>total_9</code>. Nhãn chỉ viết hoa chữ đầu khi hiển thị; <code>df_q1</code> giữ nguyên.</span>

In [19]:
from src.stats import bootstrap_diff, cohens_d

N_ROWS_BEFORE_D4 = len(df_q1)   # để kiểm tra df_q1 không đổi số dòng sau D4/D5
METHOD_ORDER = [WET, DRY]       # washed trước, natural sau


def title_labels(frame, cols):
    """Bản sao chỉ để hiển thị: viết hoa chữ đầu các cột nhãn."""
    out = frame.copy()
    for col in cols:
        out[col] = out[col].astype("string").str.title()
    return out

#### <span style="color:#3B5BA9">Bảng 1 · Chưa kiểm soát (toàn mẫu)</span>

In [20]:
d4_raw = (
    df_q1.groupby(METHOD_COL)["total_9"]
    .agg(n_lots="size", median="median", mean="mean")
    .reindex(METHOD_ORDER)
)
raw_median_diff = d4_raw.loc[DRY, "median"] - d4_raw.loc[WET, "median"]
raw_mean_diff = d4_raw.loc[DRY, "mean"] - d4_raw.loc[WET, "mean"]

table1 = d4_raw.reset_index()
table1[METHOD_COL] = table1[METHOD_COL].astype("string").str.title()
table1.loc[len(table1)] = ["Chênh lệch natural − washed", pd.NA, raw_median_diff, raw_mean_diff]
table1 = table1.rename(columns={METHOD_COL: "Phương pháp", "n_lots": "Số lô",
                                "median": "Median total_9", "mean": "Mean total_9"})
display(table1.round(2))

,Phương pháp,Số lô,Median total_9,Mean total_9
0,Washed / Wet,811,74.92,74.52
1,Natural / Dry,251,75.17,74.78
2,Chênh lệch natural − washed,<NA>,0.25,0.26


#### <span style="color:#3B5BA9">Bảng 2 · Đã kiểm soát châu lục (Châu lục × Phương pháp)</span>

In [21]:
by_cm = (
    df_q1.groupby([CONTINENT_COL, METHOD_COL])["total_9"]
    .agg(n_lots="size", median="median")
    .unstack(METHOD_COL)
)
d4_continent = pd.DataFrame({
    "n_washed": by_cm[("n_lots", WET)],
    "n_natural": by_cm[("n_lots", DRY)],
    "median_washed": by_cm[("median", WET)],
    "median_natural": by_cm[("median", DRY)],
})
d4_continent["diff_median"] = d4_continent["median_natural"] - d4_continent["median_washed"]
d4_continent["n_total"] = d4_continent["n_washed"] + d4_continent["n_natural"]
d4_continent["share_pct"] = d4_continent["n_total"] / len(df_q1) * 100

table2 = title_labels(d4_continent.rename_axis("Châu lục").reset_index(), ["Châu lục"])
table2 = table2.rename(columns={
    "n_washed": "Số lô Washed", "n_natural": "Số lô Natural",
    "median_washed": "Median Washed", "median_natural": "Median Natural",
    "diff_median": "Chênh lệch median (natural − washed)",
    "n_total": "Tổng số lô", "share_pct": "Tỷ trọng mẫu (%)",
})
display(table2.round(2))

,Châu lục,Số lô Washed,Số lô Natural,Median Washed,Median Natural,Chênh lệch median (natural − washed),Tổng số lô,Tỷ trọng mẫu (%)
0,Africa,100,27,75.46,77.01,1.55,127,11.96
1,Asia,86,27,74.62,75.25,0.62,113,10.64
2,North America,490,89,74.58,74.74,0.16,579,54.52
3,South America,135,108,75.75,75.16,-0.59,243,22.88


#### <span style="color:#3B5BA9">So sánh: chênh lệch thô vs chênh lệch trong từng châu lục</span>

<span style="color:#5B6B8C;font-size:14px"><code>Chiều so với thô</code>: dấu của chênh lệch median trong châu lục so với dấu của chênh lệch median toàn mẫu.</span>

In [22]:
def same_direction(diff, ref):
    if np.sign(diff) == 0 or np.sign(ref) == 0:
        return "Bằng 0"
    return "Cùng chiều" if np.sign(diff) == np.sign(ref) else "Ngược chiều"


d4_compare = pd.DataFrame({
    "Phạm vi": ["Toàn mẫu (thô)"] + d4_continent.index.astype("string").str.title().tolist(),
    "Số lô": [len(df_q1)] + d4_continent["n_total"].tolist(),
    "Chênh lệch median (natural − washed)": [raw_median_diff] + d4_continent["diff_median"].tolist(),
    "Chiều so với thô": ["—"] + [same_direction(d, raw_median_diff) for d in d4_continent["diff_median"]],
})
display(d4_compare.round(2))

,Phạm vi,Số lô,Chênh lệch median (natural − washed),Chiều so với thô
0,Toàn mẫu (thô),1062,0.25,—
1,Africa,127,1.55,Cùng chiều
2,Asia,113,0.62,Cùng chiều
3,North America,579,0.16,Cùng chiều
4,South America,243,-0.59,Ngược chiều


✍️ **TODO (Hoàng tự viết):** chênh lệch thô có giữ chiều trong từng châu lục không? Có dấu hiệu nghịch lý Simpson không (North America chiếm 55% mẫu)?

### <span style="color:#283E75">▍D5 · Định lượng bất định: bootstrap CI 95% và Cohen's d</span>

<span style="color:#5B6B8C;font-size:14px"><code>bootstrap_diff</code> (median, 5.000 lần, <code>seed</code> mặc định = <code>RANDOM_STATE</code>) và <code>cohens_d</code> từ <code>src/stats.py</code>; thứ tự luôn là natural − washed.</span>

In [23]:
# Đơn vị của chênh lệch median và CI: điểm total_9 (thang 0–90).
# Cohen's d không có đơn vị (chênh lệch mean / độ lệch chuẩn gộp).
scopes = [("Toàn mẫu", df_q1)] + [
    (continent, df_q1[df_q1[CONTINENT_COL].eq(continent)])
    for continent in d4_continent.index
]

d5_rows = []
for scope, part in scopes:
    washed = part.loc[part[METHOD_COL].eq(WET), "total_9"]
    natural = part.loc[part[METHOD_COL].eq(DRY), "total_9"]
    diff, (ci_lo, ci_hi) = bootstrap_diff(natural, washed)
    d5_rows.append({
        "Phạm vi": scope,
        "n washed": len(washed),
        "n natural": len(natural),
        "Chênh lệch median": diff,
        "CI thấp": ci_lo,
        "CI cao": ci_hi,
        "CI chứa 0?": bool(ci_lo <= 0 <= ci_hi),
        "Cohen's d": cohens_d(natural, washed),
    })

d5_table = pd.DataFrame(d5_rows).round(2)

d5_display = d5_table.copy()
is_continent = d5_display["Phạm vi"].ne("Toàn mẫu")
d5_display.loc[is_continent, "Phạm vi"] = d5_display.loc[is_continent, "Phạm vi"].str.title()
display(d5_display)

,Phạm vi,n washed,n natural,Chênh lệch median,CI thấp,CI cao,CI chứa 0?,Cohen's d
0,Toàn mẫu,811,251,0.25,0.07,0.50,False,0.11
1,Africa,100,27,1.55,0.66,2.51,False,0.88
2,Asia,86,27,0.62,-0.47,1.42,True,0.12
3,North America,490,89,0.16,-0.63,0.65,True,-0.04
4,South America,135,108,-0.59,-0.98,-0.33,False,-0.37


✍️ **TODO (Hoàng tự viết):** diễn giải CI và Cohen's d — châu lục nào khác biệt rõ, châu lục nào không đủ bằng chứng; độ lớn thực tế tính bằng điểm.

In [24]:
# TEST
assert d4_continent["n_total"].sum() == len(df_q1) == 1062, (
    f"Tổng số lô Bảng 2 = {d4_continent['n_total'].sum()} khác len(df_q1) = {len(df_q1)} (mong đợi 1062)"
)

assert len(d5_table) == 1 + len(d4_continent) == 5, (
    f"Bảng D5 phải có 5 dòng (toàn mẫu + 4 châu lục), đang có {len(d5_table)}"
)
d5_raw = pd.DataFrame(d5_rows)   # giá trị chưa làm tròn
bad_ci = d5_raw[~(d5_raw["CI thấp"].le(d5_raw["Chênh lệch median"])
                  & d5_raw["Chênh lệch median"].le(d5_raw["CI cao"]))]
assert bad_ci.empty, f"Chênh lệch median nằm ngoài CI ở:\n{bad_ci}"

overall_observed = d5_raw.loc[d5_raw["Phạm vi"].eq("Toàn mẫu"), "Chênh lệch median"].iloc[0]
assert np.isclose(raw_median_diff, overall_observed), (
    f"Chênh lệch median Bảng 1 ({raw_median_diff}) khác observed của bootstrap_diff ({overall_observed})"
)

assert len(df_q1) == N_ROWS_BEFORE_D4, (
    f"df_q1 đổi số dòng sau D4/D5: {N_ROWS_BEFORE_D4} → {len(df_q1)}"
)
print("Q1 D4/D5 checks passed")

Q1 D4/D5 checks passed


### <span style="color:#283E75">▍Lưu dữ liệu đã xử lý</span>

In [25]:
PROCESSED.mkdir(parents=True, exist_ok=True)
Q1_CLEAN_FILE = PROCESSED / "q1_clean.csv"
df_q1.to_csv(Q1_CLEAN_FILE, index=False)
print(f"Đã lưu {len(df_q1):,} lô × {df_q1.shape[1]} cột → {Q1_CLEAN_FILE.relative_to(PROJECT_ROOT)}")

Đã lưu 1,062 lô × 23 cột → data/processed/q1_clean.csv


### <span style="color:#283E75">▍Nhận xét</span>

✍️ **TODO (Hoàng tự viết):** nhận xét số lô bị loại ở mỗi bước và lý do

✍️ **TODO (Hoàng tự viết):** biện luận D3 — vì sao dùng total_9 thay vì từng thuộc tính